In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    IntegerType,
    StringType,
    DateType,
    BooleanType,
    DecimalType
)

# ============================================================
# STEP 1: CONFIGURATION
# ============================================================

catalog = "databrickscaresync"

source_table = f"{catalog}.silver.lab_results"
hospital_table = f"{catalog}.silver.hospitals"
calendar_table = f"{catalog}.silver.calendar"

target_table = f"{catalog}.gold.hospital_daily_summary"

print("========== HOSPITAL DAILY SUMMARY ==========")
print("[SOURCE]", source_table)
print("[HOSPITAL]", hospital_table)
print("[CALENDAR]", calendar_table)
print("[TARGET]", target_table)


# ============================================================
# STEP 2: VALIDATE SOURCE TABLES
# ============================================================

for table_name in [
    source_table,
    hospital_table,
    calendar_table
]:
    if not spark.catalog.tableExists(table_name):
        raise ValueError(
            f"Required source table does not exist: {table_name}"
        )

lab_df = spark.table(source_table)
hospital_df = spark.table(hospital_table)
calendar_df = spark.table(calendar_table)

required_lab_columns = [
    "hospital_id",
    "patient_id",
    "doctor_id",
    "test_date",
    "result_status"
]

required_hospital_columns = [
    "hospital_id",
    "hospital_name"
]

required_calendar_columns = [
    "date_key",
    "date",
    "day_name",
    "is_weekend"
]


def validate_columns(df, required_columns, table_name):
    missing = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{table_name} is missing columns: {missing}"
        )


validate_columns(
    lab_df,
    required_lab_columns,
    source_table
)

validate_columns(
    hospital_df,
    required_hospital_columns,
    hospital_table
)

validate_columns(
    calendar_df,
    required_calendar_columns,
    calendar_table
)

print("[VALIDATION PASSED] Source schemas are valid.")


# ============================================================
# STEP 3: VALIDATE AND PREPARE HOSPITAL DATA
# ============================================================
# Silver may contain repeated hospital records.
# We do not modify Silver.
#
# We allow repeated IDs only when each ID has one consistent
# hospital_name. Conflicting names stop the pipeline.

if hospital_df.filter(
    F.col("hospital_id").isNull()
).limit(1).count() > 0:
    raise ValueError(
        "Silver hospitals contains NULL hospital_id values."
    )

hospital_name_conflicts = (
    hospital_df
    .groupBy("hospital_id")
    .agg(
        F.countDistinct("hospital_name")
        .alias("distinct_hospital_names")
    )
    .filter(F.col("distinct_hospital_names") > 1)
)

if hospital_name_conflicts.limit(1).count() > 0:
    print("[ERROR] Conflicting hospital names found:")
    hospital_name_conflicts.show(20, truncate=False)

    raise ValueError(
        "A hospital_id has multiple different hospital names. "
        "Resolve the source data before building the Gold table."
    )

hospital_duplicates = (
    hospital_df
    .groupBy("hospital_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "[HOSPITAL] Duplicate hospital IDs:",
    hospital_duplicates.count()
)

# Safe for the join because conflicting names were checked above.
# This removes repeated dimension rows from this DataFrame only.

hospital_lookup = (
    hospital_df
    .select("hospital_id", "hospital_name")
    .dropDuplicates(["hospital_id", "hospital_name"])
)

if hospital_lookup.groupBy("hospital_id").count().filter(
    F.col("count") > 1
).limit(1).count() > 0:
    raise ValueError(
        "Hospital lookup is not unique by hospital_id."
    )

print(
    "[HOSPITAL] Unique hospital records:",
    hospital_lookup.count()
)


# ============================================================
# STEP 4: VALIDATE CALENDAR DATA
# ============================================================

if calendar_df.filter(
    F.col("date").isNull()
).limit(1).count() > 0:
    raise ValueError(
        "Calendar contains NULL dates."
    )

calendar_lookup = (
    calendar_df
    .select(
        F.to_date("date").alias("calendar_date"),
        "day_name",
        "is_weekend"
    )
)

calendar_duplicates = (
    calendar_lookup
    .groupBy("calendar_date")
    .count()
    .filter(F.col("count") > 1)
)

if calendar_duplicates.limit(1).count() > 0:
    print("[ERROR] Duplicate calendar dates:")
    calendar_duplicates.show(20, truncate=False)

    raise ValueError(
        "Calendar contains duplicate dates. "
        "Calendar join could duplicate summary rows."
    )

print("[VALIDATION PASSED] Calendar dates are unique.")


# ============================================================
# STEP 5: VALIDATE LAB RESULTS
# ============================================================

if lab_df.filter(
    F.col("hospital_id").isNull() |
    F.col("patient_id").isNull() |
    F.col("doctor_id").isNull() |
    F.col("test_date").isNull()
).limit(1).count() > 0:
    raise ValueError(
        "Lab results contain NULL values in required fields."
    )

lab_df = lab_df.withColumn(
    "test_date",
    F.to_date("test_date")
)

if lab_df.filter(
    F.col("test_date").isNull()
).limit(1).count() > 0:
    raise ValueError(
        "Some test_date values could not be converted to dates."
    )

if lab_df.limit(1).count() == 0:
    raise ValueError(
        "Lab results source is empty."
    )

print("[VALIDATION PASSED] Lab results data is available.")


# ============================================================
# STEP 6: AGGREGATE LAB RESULTS
# Grain: One row per hospital per day
# ============================================================

daily_df = (
    lab_df
    .groupBy(
        "hospital_id",
        "test_date"
    )
    .agg(
        F.count("*").alias("total_tests"),

        F.countDistinct("patient_id")
        .alias("unique_patients_tested"),

        F.countDistinct("doctor_id")
        .alias("active_doctor_count"),

        F.sum(
            F.when(
                F.col("result_status") == "Abnormal",
                1
            ).otherwise(0)
        ).alias("abnormal_test_count"),

        F.sum(
            F.when(
                F.col("result_status") == "Critical",
                1
            ).otherwise(0)
        ).alias("critical_test_count")
    )
)

print("[STEP 6 COMPLETE] Lab results aggregated.")


# ============================================================
# STEP 7: CALCULATE ABNORMAL AND CRITICAL RATES
# ============================================================

daily_df = (
    daily_df
    .withColumn(
        "abnormal_rate_pct",
        F.round(
            F.col("abnormal_test_count") /
            F.col("total_tests") * 100,
            2
        ).cast(DecimalType(5, 2))
    )
    .withColumn(
        "critical_rate_pct",
        F.round(
            F.col("critical_test_count") /
            F.col("total_tests") * 100,
            2
        ).cast(DecimalType(5, 2))
    )
)


# ============================================================
# STEP 8: JOIN HOSPITAL INFORMATION
# ============================================================

daily_df = (
    daily_df.alias("d")
    .join(
        hospital_lookup.alias("h"),
        F.col("d.hospital_id") == F.col("h.hospital_id"),
        "left"
    )
    .select(
        F.col("d.hospital_id").cast("int").alias("hospital_id"),
        F.col("h.hospital_name").alias("hospital_name"),
        F.col("d.test_date").alias("summary_date"),
        F.col("d.total_tests"),
        F.col("d.unique_patients_tested"),
        F.col("d.active_doctor_count"),
        F.col("d.abnormal_test_count"),
        F.col("d.critical_test_count"),
        F.col("d.abnormal_rate_pct"),
        F.col("d.critical_rate_pct")
    )
)

print("[STEP 8 COMPLETE] Hospital information joined.")


# ============================================================
# STEP 9: JOIN CALENDAR INFORMATION
# ============================================================

daily_df = (
    daily_df.alias("d")
    .join(
        calendar_lookup.alias("c"),
        F.col("d.summary_date") == F.col("c.calendar_date"),
        "left"
    )
    .select(
        "d.*",
        F.col("c.day_name").alias("day_name"),
        F.col("c.is_weekend").alias("is_weekend")
    )
)


# ============================================================
# STEP 10: CALCULATE PREVIOUS DAY TEST VOLUME
# ============================================================

window_spec = (
    Window
    .partitionBy("hospital_id")
    .orderBy("summary_date")
)

daily_df = (
    daily_df
    .withColumn(
        "prev_day_total_tests",
        F.lag("total_tests").over(window_spec)
    )
    .withColumn(
        "test_volume_change_pct",
        F.when(
            F.col("prev_day_total_tests").isNull() |
            (F.col("prev_day_total_tests") == 0),
            F.lit(None)
        ).otherwise(
            F.round(
                (
                    F.col("total_tests") -
                    F.col("prev_day_total_tests")
                ) / F.col("prev_day_total_tests") * 100,
                2
            )
        ).cast(DecimalType(6, 2))
    )
)


# ============================================================
# STEP 11: SELECT FINAL STTM COLUMNS
# ============================================================

final_df = daily_df.select(
    F.col("hospital_id").cast(IntegerType()),
    F.col("hospital_name").cast(StringType()),
    F.col("summary_date").cast(DateType()),
    F.col("day_name").cast(StringType()),
    F.col("is_weekend").cast(BooleanType()),
    F.col("total_tests").cast(IntegerType()),
    F.col("unique_patients_tested").cast(IntegerType()),
    F.col("active_doctor_count").cast(IntegerType()),
    F.col("abnormal_test_count").cast(IntegerType()),
    F.col("critical_test_count").cast(IntegerType()),
    F.col("abnormal_rate_pct").cast(DecimalType(5, 2)),
    F.col("critical_rate_pct").cast(DecimalType(5, 2)),
    F.col("prev_day_total_tests").cast(IntegerType()),
    F.col("test_volume_change_pct").cast(DecimalType(6, 2))
)


# ============================================================
# STEP 12: VALIDATE FINAL DATA
# ============================================================

if final_df.limit(1).count() == 0:
    raise ValueError(
        "Final daily summary is empty. No target table was modified."
    )

duplicate_grain = (
    final_df
    .groupBy("hospital_id", "summary_date")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_grain.limit(1).count() > 0:
    duplicate_grain.show(20, truncate=False)

    raise ValueError(
        "Duplicate hospital_id + summary_date records found. "
        "No target table was modified."
    )

print("[FINAL VALIDATION PASSED]")
print("Rows to write:", final_df.count())

print("========== FINAL DATA PREVIEW ==========")
final_df.orderBy(
    "hospital_id",
    "summary_date"
).show(10, truncate=False)


# ============================================================
# STEP 13: WRITE GOLD TARGET TABLE
# ============================================================
# Creates the Gold table only if it does not already exist.
# Does not overwrite or delete an existing Gold table.

if spark.catalog.tableExists(target_table):
    raise RuntimeError(
        f"Target already exists: {target_table}. "
        "Existing Gold data has been protected. "
        "No overwrite was performed."
    )

(
    final_df.write
    .format("delta")
    .mode("error")  # same as errorifexists; Spark Connect only accepts "error"
    .saveAsTable(target_table)
)

print(f"[DONE] Created {target_table}")


# ============================================================
# STEP 14: POST-WRITE VALIDATION
# ============================================================

result_df = spark.table(target_table)

print("\n========== FINAL TABLE VALIDATION ==========")

result_df.printSchema()

result_df.select(
    F.count("*").alias("total_rows"),
    F.countDistinct(
        "hospital_id",
        "summary_date"
    ).alias("distinct_hospital_days"),
    F.sum("total_tests").alias("total_tests")
).show()

print("========== SAMPLE GOLD RECORDS ==========")

result_df.orderBy(
    "hospital_id",
    "summary_date"
).show(10, truncate=False)

print("========== PIPELINE COMPLETED ==========")


In [0]:

from pyspark.sql import functions as F

catalog = "databrickscaresync"

tables_to_check = [
    f"{catalog}.silver.lab_results",
    f"{catalog}.bronze.lab_results",
    f"{catalog}.silver.hospitals",
    f"{catalog}.bronze.hospitals"
]

print("========== LAB RESULTS PIPELINE DIAGNOSTIC ==========")

for table_name in tables_to_check:

    print(f"\n========== {table_name} ==========")

    if spark.catalog.tableExists(table_name):

        df = spark.table(table_name)

        print("Table exists: YES")
        print("Record count:", df.count())
        print("Columns:", df.columns)

        if "landing_timestamp" in df.columns:
            print("Landing batches:")
            display(
                df.groupBy("landing_timestamp")
                .count()
                .orderBy("landing_timestamp")
            )

        if df.limit(1).count() > 0:
            print("Sample records:")
            display(df.limit(5))
        else:
            print("WARNING: Table is EMPTY.")

    else:
        print("Table exists: NO")

print("\n========== DIAGNOSTIC COMPLETED ==========")
print("No tables were modified.")


In [0]:
updated_df= spark.table("databrickscaresync.gold.hospital_daily_summary")
record_count =updated_df.count()
dbutils.notebook.exit(str(record_count))